# Praktikum 3

In [1]:
import numpy as np
import pandas as pd

raw_df = pd.read_csv('burnout_dataset1.csv')
raw_df.head()

,Person_ID,Age,Gender,Country,Occupation,Education_Level,Employment_Status,Monthly_Income_USD,Work_Hours_Per_Week,Remote_Work,...,Family_History_Mental_Illness,Therapy_Attendance,Support_System,Life_Satisfaction,Productivity_Score,Absenteeism_Days,Burnout_Score,Mental_Health_Status,AI_Wellness_Recommendation,Burnout_Risk
0,1,58.0,Male,United States,Marketing Manager,Diploma,Employed,2786.0,67,Hybrid,...,No,Yes,Average,6.0,19.0,2,82,Critical,Practice Mindfulness,High
1,2,24.0,Female,Pakistan,Freelancer,Bachelor,Student,8027.0,44,Hybrid,...,Yes,No,Good,4.0,27.0,22,64,Critical,Reduce Workload,Moderate
2,3,59.0,Male,Malaysia,Accountant,PhD,Employed,3177.0,60,No,...,Yes,No,Average,4.0,51.0,17,50,Needs Attention,Increase Physical Activity,Moderate
3,4,65.0,Other,Mexico,Chef,Master,Self-Employed,4093.0,25,Yes,...,No,Yes,Good,7.0,83.0,14,16,Healthy,Maintain Healthy Routine,Low
4,5,64.0,Other,Japan,Farmer,Diploma,Student,NaN,52,No,...,Yes,No,Poor,9.0,61.0,4,26,Healthy,Reduce Workload,Low


In [2]:
# Membentuk representasi teks profil karyawan dari kolom kategorikal dan rekomendasi
raw_df['Profil_Teks'] = (
    'occupation_' + raw_df['Occupation'].fillna('unknown').astype(str) + ' ' +
    'stress_' + raw_df['Stress_Level'].fillna('unknown').astype(str) + ' ' +
    'sleep_' + raw_df['Sleep_Quality'].fillna('unknown').astype(str) + ' ' +
    'chronic_' + raw_df['Chronic_Stress'].fillna('unknown').astype(str) + ' ' +
    'exercise_' + raw_df['Exercise_Frequency'].fillna('unknown').astype(str) + ' ' +
    'diet_' + raw_df['Healthy_Diet'].fillna('unknown').astype(str) + ' ' +
    'support_' + raw_df['Support_System'].fillna('unknown').astype(str) + ' ' +
    'remote_' + raw_df['Remote_Work'].fillna('unknown').astype(str) + ' ' +
    'alcohol_' + raw_df['Alcohol_Consumption'].fillna('unknown').astype(str) + ' ' +
    'smoking_' + raw_df['Smoking'].fillna('unknown').astype(str) + ' ' +
    raw_df['AI_Wellness_Recommendation'].fillna('').astype(str)
)

df = raw_df[['Burnout_Risk', 'Profil_Teks']].copy()
df.head()

,Burnout_Risk,Profil_Teks
0,High,occupation_Marketing Manager stress_High sleep...
1,Moderate,occupation_Freelancer stress_Moderate sleep_Po...
2,Moderate,occupation_Accountant stress_Moderate sleep_Av...
3,Low,occupation_Chef stress_Low sleep_Good chronic_...
4,Low,occupation_Farmer stress_Low sleep_Excellent c...


In [3]:
# Ubah nama kolom label dan teks profil agar selaras dengan modul
df = df.rename(columns={'Burnout_Risk': 'Labels', 'Profil_Teks': 'Profil'})
df['Labels'] = df['Labels'].apply(lambda x: 'High' if x == 'High' else 'Non-High')
df.head()

,Labels,Profil
0,High,occupation_Marketing Manager stress_High sleep...
1,Non-High,occupation_Freelancer stress_Moderate sleep_Po...
2,Non-High,occupation_Accountant stress_Moderate sleep_Av...
3,Non-High,occupation_Chef stress_Low sleep_Good chronic_...
4,Non-High,occupation_Farmer stress_Low sleep_Excellent c...


In [4]:
# Cek Jumlah Data Per Kelas
print(df['Labels'].value_counts())
print('\n')

# Cek Kelengkapan Data
print(df.info())
print('\n')

# Cek Statistik Deskriptif
print(df.describe())

Labels
Non-High    40149
High         9851
Name: count, dtype: int64


<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Labels  50000 non-null  str  
 1   Profil  50000 non-null  str  
dtypes: str(2)
memory usage: 8.4 MB
None


          Labels                                             Profil
count      50000                                              50000
unique         2                                              47579
top     Non-High  occupation_Software Engineer stress_Moderate s...
freq       40149                                                  4


In [5]:
# Data untuk label
new_labels = {
    'High': 1,
    'Non-High': 0
}

# Encode label
df['Labels'] = df['Labels'].map(new_labels)

# Cek data
df.head()

,Labels,Profil
0,1,occupation_Marketing Manager stress_High sleep...
1,0,occupation_Freelancer stress_Moderate sleep_Po...
2,0,occupation_Accountant stress_Moderate sleep_Av...
3,0,occupation_Chef stress_Low sleep_Good chronic_...
4,0,occupation_Farmer stress_Low sleep_Excellent c...


In [6]:
X = df['Profil'].to_numpy()

y = df['Labels'].to_numpy()

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# Split data training dan testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

# Inisiasi CountVectorizer
bow = CountVectorizer()

# Fitting dan transform X_train dengan CountVectorizer
X_train = bow.fit_transform(X_train)

# Transform X_test saja agar tidak terjadi kebocoran informasi dari data uji
X_test = bow.transform(X_test)

In [8]:
print(len(bow.get_feature_names_out()))
print(f'Dimensi data: {X_train.shape}')

71
Dimensi data: (40000, 71)


In [9]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Inisiasi MultinomialNB
mnb = MultinomialNB()

# Fit model
mnb.fit(X_train, y_train)

# Prediksi dengan data training
y_pred_train = mnb.predict(X_train)

# Evaluasi akurasi data training
acc_train = accuracy_score(y_train, y_pred_train)

# Prediksi dengan data testing
y_pred_test = mnb.predict(X_test)

# Evaluasi akurasi data testing
acc_test = accuracy_score(y_test, y_pred_test)

# Print hasil evaluasi
print(f'Hasil akurasi data train: {acc_train}')
print(f'Hasil akurasi data test: {acc_test}')

Hasil akurasi data train: 0.918175
Hasil akurasi data test: 0.9185
